# Block 15: The full encoder-decoder of Figure 1

**Paper anchor:** Figure 1, §3.1 (decoder's third sub-layer), §3.2.3 (first bullet: encoder-decoder attention), 'outputs shifted right'
**Builds on:** blocks 09 and 10
**Time:** about 8 hours, in short sessions
**You will write:** `shift_right`, `TransformerSeq2Seq`
**Done when:** both checkpoints pass and you can redraw Figure 1 with every shape

> **How this notebook works.** Claude teaches each concept in chat, one step at a time (type `next`).
> You write every line of solution code, in the cells marked **YOUR CODE**. Checkpoint cells test behavior only
> and never contain a solution (please don't open `checks/` until you pass). Stuck or wrong? Paste your code and
> the output into chat: we find the misconception together. Log every mistake in the table at the bottom.

In [ ]:
# provided: repo paths (works from notebooks/ locally and from the repo root on Colab)
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
np.set_printoptions(precision=4, suppress=True)
from checks import b15
import torch, torch.nn as nn
from minitransformer.attention import MultiHeadAttention
from minitransformer.layers import FeedForward, TransformerBlock
from minitransformer.numpy_ops import sinusoidal_pe

## Big picture (Pass 2 for the whole of Fig. 1)
The encoder reads the source with unmasked self-attention. Each decoder layer adds a third sub-layer between its masked
self-attention and its FFN: queries from the decoder, keys and values from the encoder output (§3.2.3).

| step | shape |
|---|---|
| src | (B, S) |
| encoder output, "memory" | (B, S, d) |
| tgt_in (shifted right) | (B, T) |
| decoder self-attention mask: causal | (T, T) |
| cross-attention mask: source padding | (B, 1, 1, S) |
| logits | (B, T, V_tgt) |

**Traps.** PAD tokens must be masked as keys in encoder self-attention and in cross-attention. Target padding sits at
the end, so the causal mask already hides it from real positions; ignore PAD targets in the loss (`ignore_index`).

## Before you code
Answer in the cell below, in your own words. Guessing is fine; we check in chat.

1. Why does the decoder see the target 'shifted right' during training?
2. In cross-attention, which tensor supplies Q and which supplies K and V? What shape are the weights?
3. What would go wrong if PAD tokens were not masked in the source?

*My answers:*

1. 
2. 
3. 

In [ ]:
# YOUR CODE: replace each `raise NotImplementedError` with your implementation

def shift_right(tgt, bos_id):
    """Decoder input for teacher forcing (Fig. 1, "Outputs (shifted right)").

    tgt: (B, T) int64 target ids. returns (B, T): bos_id, then tgt[:, :-1].
    """
    raise NotImplementedError("Block 15: your turn")

class TransformerSeq2Seq(nn.Module):
    """Encoder-decoder Transformer, Fig. 1 (post-norm, §3.1).

    __init__(src_vocab, tgt_vocab, d_model, n_heads, n_enc_layers, n_dec_layers, d_ff, max_len,
             dropout=0.1, pad_id=0)
        Encoder layer: self-attention + FFN.  Decoder layer: masked self-attention,
        encoder-decoder attention (queries from the decoder, keys/values from the encoder output,
        §3.2.3), FFN. Each sub-layer wrapped in residual + LayerNorm. Sinusoidal positions.
        Embeddings scaled by sqrt(d_model). Output projection to tgt_vocab.
        Sharing embeddings across languages (§3.4) needs a shared vocabulary; optional here.

    encode(src) -> (memory, src_mask)
        src: (B, S) int64, may contain pad_id
        memory: (B, S, d_model); src_mask: bool (B, 1, 1, S), True where src != pad_id
    decode(tgt_in, memory, src_mask) -> logits (B, T, tgt_vocab)
        tgt_in: (B, T) int64 decoder input (already shifted right)
        causal mask on decoder self-attention; src_mask on encoder-decoder attention.
    encode must ALSO apply src_mask in encoder self-attention, so PAD tokens never influence
    real positions (appending PADs to a source must not change the output).
    forward(src, tgt_in) == decode(tgt_in, *encode(src))
    """

    def __init__(self, src_vocab, tgt_vocab, d_model, n_heads, n_enc_layers, n_dec_layers, d_ff, max_len,
                 dropout=0.1, pad_id=0):
        super().__init__()
        raise NotImplementedError("Block 15: your turn")

    def encode(self, src):
        raise NotImplementedError("Block 15: your turn")

    def decode(self, tgt_in, memory, src_mask):
        raise NotImplementedError("Block 15: your turn")

    def forward(self, src, tgt_in):
        raise NotImplementedError("Block 15: your turn")

In [ ]:
b15.check_shift_right(shift_right)
b15.check_seq2seq(TransformerSeq2Seq)

**Manual check (the automatic checks cannot see it):** confirm that both your source and target embeddings are multiplied by sqrt(d_model) (§3.4).

## Explain it back
Without notes. Then paste your answers into chat.

1. Redraw Figure 1 from memory and annotate every arrow with its shape.
2. What exactly differs between an encoder layer and a decoder layer?
3. Why can the encoder be computed once per sentence while the decoder runs once per output token at inference?

*My explanation:*

1. 
2. 
3. 

## Mistakes log
Every wrong attempt is data. Copy finished rows into `journal/mistakes.md`.

| # | What I wrote | Symptom | The misconception | The fix |
|---|---|---|---|---|
| 1 | | | | |

## Graduate
Copy into `src/minitransformer/seq2seq.py`; `pytest -q`; commit `block 15: encoder-decoder`.